# Stage 7 - Cascade, corrected

**Standalone.** Regenerates the end-to-end results with the side-head fix and
saves the per-class confusion matrix, which the original run never wrote.

### Why this exists

`stage7_confusion.csv` currently holds the **pre-fix** run: the one where
`train_det` computed only the contact loss while inference still used the side
head to decide which player to classify. Roughly half the classifier windows
were cut around the wrong player, and macro-F1 came out at 0.481.

The corrected run reached 0.762, but its fix cell wrote to `stage7_v2_sweep.csv`
and `stage7_v2_per_video.csv` and never overwrote the confusion file. So any
table reading `stage7_confusion.csv` reports the bug as though it were the cost
of cascading.

This re-runs with the side loss included and saves:

    stage7_v2_confusion.csv       per-class confusion, corrected
    stage7_v2_per_class.csv       precision / recall / F1 / support
    stage7_v2_sweep.csv           tolerance sweep
    stage7_v2_per_video.csv       per-video breakdown

**GPU required, ~15 min.**


## 1. Mount

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Config

In [2]:
BASE = "/content/drive/MyDrive/tt_coach"

EXCLUDE   = ["test_5"]          # no contact signal in the pose stream
TOLS      = [1, 2, 3, 5, 8, 12, 20]
SEED      = 42
CLS_EPOCHS = 60
DET_EPOCHS = 40
WINDOW_CLS = 97                 # classifier window, from folds.json
BATCH     = 64

import json, math, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore")

BASE = Path(BASE); META = BASE/"derived/meta"; CLIP = BASE/"derived/clips"
STREAM = BASE/"derived/pose_stream"; OUT = BASE/"outputs"
(OUT/"metrics").mkdir(parents=True, exist_ok=True)
(OUT/"figures").mkdir(parents=True, exist_ok=True)

dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "No GPU. Runtime > Change runtime type > T4."
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def load(stem):
    p = META/f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META/f"{stem}.csv")

strokes = load("strokes")
folds   = json.loads((META/"folds.json").read_text())
V2F, PRE, NF = folds["video2fold"], folds["window"]["pre"], folds["window"]["n_frames"]
CLASSES = ["serve", "attack", "control", "defence"]
C2I = {c: i for i, c in enumerate(CLASSES)}

VIDEOS = [v for v in sorted(strokes.video_id.unique(),
          key=lambda x: (x.split("_")[0], int(x.split("_")[1])))
          if v not in EXCLUDE]
FOLDS = sorted({V2F[v] for v in VIDEOS})

# ground truth: frame -> (class, side) per video
GT = {v: {} for v in VIDEOS}
for r in strokes.itertuples():
    if r.video_id in GT:
        GT[r.video_id][int(r.frame_120)] = (r.shot_class, r.side)

print(f"{len(VIDEOS)} videos ({len(EXCLUDE)} excluded), {len(FOLDS)} folds")
print(f"contacts in scope: {sum(len(g) for g in GT.values())}")

11 videos (1 excluded), 7 folds
contacts in scope: 1430


## 3. Pose streams

In [3]:
L_SHO, R_SHO, L_WRI, R_WRI, L_HIP, R_HIP = 5, 6, 9, 10, 11, 12
FLIP = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]

def canon(kp, sc, seg, mirror):
    kp = kp.astype(np.float32).copy()
    hip = (kp[:, L_HIP] + kp[:, R_HIP]) / 2
    sho = (kp[:, L_SHO] + kp[:, R_SHO]) / 2
    torso = np.linalg.norm(sho - hip, axis=-1)
    scale = np.ones(len(kp), np.float32)
    for s in np.unique(seg):
        m = seg == s; t = torso[m]; t = t[t > 1]
        scale[m] = np.median(t) if len(t) else 1.0
    kp = (kp - hip[:, None, :]) / np.maximum(scale, 1e-3)[:, None, None]
    if mirror:
        kp[..., 0] *= -1
        sc = sc.copy()
        for a, b in FLIP:
            kp[:, [a, b]] = kp[:, [b, a]]; sc[:, [a, b]] = sc[:, [b, a]]
    return kp, sc


S = {}
for v in VIDEOS:
    d = np.load(STREAM/f"{v}.npz", allow_pickle=True)
    fidx, seg = d["frame_idx"], d["seg_id"]
    KP, SC, DET = d["keypoints"], d["scores"], d["detected"]
    ch, kps, vals = [], [], []
    for pi in (0, 1):
        kp, sc = canon(KP[:, pi], SC[:, pi].astype(np.float32), seg, mirror=(pi == 1))
        vel = np.zeros_like(kp); vel[1:] = np.diff(kp, axis=0)
        vel[np.diff(seg, prepend=seg[0]) != 0] = 0
        ch += [kp.reshape(len(kp), -1), vel.reshape(len(kp), -1),
               (sc * DET[:, pi:pi+1]).astype(np.float32)]
        kps.append(kp); vals.append((sc >= 0.35) & DET[:, pi:pi+1])
    X = np.nan_to_num(np.concatenate(ch, 1).astype(np.float32))

    pos = np.array([i for i in np.where(d["is_contact"])[0]
                    if int(fidx[i]) in GT[v]], int)
    yc = np.zeros(len(fidx), np.float32); t = np.arange(len(fidx))
    for i in pos:
        lo, hi = max(0, i-8), min(len(fidx), i+9)
        yc[lo:hi] = np.maximum(yc[lo:hi], np.exp(-((t[lo:hi]-i)**2)/8.0))

    cuts = np.where(np.diff(seg) != 0)[0] + 1
    b = np.concatenate([[0], cuts, [len(seg)]])
    S[v] = dict(X=X, yc=yc, pos=pos, fidx=fidx, seg=seg,
                kp=np.stack(kps, 1), val=np.stack(vals, 1),
                spans=[(int(b[i]), int(b[i+1])) for i in range(len(b)-1)],
                fold=V2F[v])
    print(f"  {v}: {len(X):,} frames, {len(pos)} contacts")

C_DET = S[VIDEOS[0]]["X"].shape[1]
print(f"\ndetector input channels: {C_DET}")

  game_1: 20,144 frames, 161 contacts
  game_2: 70,873 frames, 399 contacts
  game_3: 27,911 frames, 153 contacts
  game_4: 24,249 frames, 173 contacts
  game_5: 34,004 frames, 248 contacts
  test_1: 8,027 frames, 84 contacts
  test_2: 2,806 frames, 29 contacts
  test_3: 4,070 frames, 24 contacts
  test_4: 13,251 frames, 71 contacts
  test_6: 5,335 frames, 39 contacts
  test_7: 5,760 frames, 49 contacts

detector input channels: 170


## 4. Architectures

In [4]:
class Block(nn.Module):
    def __init__(s, c, d, drop=0.1):
        super().__init__()
        s.c1 = nn.Conv1d(c, c, 5, padding=2*d, dilation=d)
        s.c2 = nn.Conv1d(c, c, 5, padding=2*d, dilation=d)
        s.n1, s.n2 = nn.BatchNorm1d(c), nn.BatchNorm1d(c)
        s.do = nn.Dropout(drop)
    def forward(s, x):
        r = x
        x = s.do(F.gelu(s.n1(s.c1(x))))
        x = s.do(F.gelu(s.n2(s.c2(x))))
        return F.gelu(x + r)

class DetNet(nn.Module):
    def __init__(s, c_in, w=128):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv1d(c_in, w, 1), nn.BatchNorm1d(w), nn.GELU())
        s.blocks = nn.Sequential(*[Block(w, d) for d in (1,2,4,8,16,32,64)])
        s.hc, s.hs = nn.Conv1d(w, 1, 1), nn.Conv1d(w, 1, 1)
    def forward(s, x):
        z = s.blocks(s.stem(x))
        return s.hc(z).squeeze(1), s.hs(z).squeeze(1)

class AttnPool(nn.Module):
    def __init__(s, c):
        super().__init__(); s.score = nn.Conv1d(c, 1, 1)
    def forward(s, x):
        w = torch.softmax(s.score(x), -1)
        return torch.cat([(x*w).sum(-1), x.max(-1).values], -1)

class ClsNet(nn.Module):
    def __init__(s, c_in, w=128, n_tech=8):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv1d(c_in, w, 1), nn.BatchNorm1d(w), nn.GELU())
        s.blocks = nn.Sequential(*[Block(w, d, 0.2) for d in (1,2,4,8,16,32)])
        s.pool = AttnPool(w)
        s.trunk = nn.Sequential(nn.Linear(w*2, 256), nn.GELU(), nn.Dropout(0.3))
        s.shot, s.tech = nn.Linear(256, 4), nn.Linear(256, n_tech)
    def forward(s, x):
        z = s.trunk(s.pool(s.blocks(s.stem(x))))
        return s.shot(z), s.tech(z)

def focal(lg, tg, weight=None, g=2.0):
    m = tg >= 0
    if m.sum() == 0: return lg.sum()*0.0
    lg, tg = lg[m], tg[m]
    ce = F.cross_entropy(lg, tg, weight=weight, reduction="none")
    pt = torch.exp(-F.cross_entropy(lg, tg, reduction="none"))
    return ((1-pt)**g * ce).mean()

print("models defined")

models defined


## 5. Classifier windows

In [5]:
TECHS = ["block","chop","flick","lob","loop","push","serve","smash"]
T2I = {t: i for i, t in enumerate(TECHS)}

def window_at(v, idx, side):
    """97-frame classifier input centred on stream index `idx`."""
    d = S[v]
    lo, hi = idx - PRE, idx - PRE + NF
    sel = np.clip(np.arange(lo, hi), 0, len(d["X"]) - 1)
    pi = 0 if side == "left" else 1
    kp = d["kp"][sel, pi]                       # (T,17,2) already canonical
    val = d["val"][sel, pi].astype(np.float32)
    vel = np.zeros_like(kp); vel[1:] = np.diff(kp, axis=0)
    x = np.concatenate([kp.reshape(NF, -1), vel.reshape(NF, -1), val,
                        np.zeros((NF, 1), np.float32)], 1)
    return np.nan_to_num(x).T.astype(np.float32)

C_CLS = window_at(VIDEOS[0], S[VIDEOS[0]]["pos"][0], "left").shape[0]
print(f"classifier input: {C_CLS} channels x {NF} frames")

# training set for the classifier: ground-truth windows from the stream
CLS_X, CLS_Y, CLS_T, CLS_V = [], [], [], []
for v in VIDEOS:
    d = S[v]
    for i in d["pos"]:
        f = int(d["fidx"][i]); cls, side = GT[v][f]
        CLS_X.append(window_at(v, i, side)); CLS_Y.append(C2I[cls])
        tech = strokes[(strokes.video_id == v) &
                       (strokes.frame_120 == f)].technique.iloc[0]
        CLS_T.append(T2I.get(tech, -1)); CLS_V.append(v)
CLS_X = np.stack(CLS_X); CLS_Y = np.array(CLS_Y)
CLS_T = np.array(CLS_T); CLS_V = np.array(CLS_V)
CLS_F = np.array([V2F[v] for v in CLS_V])
print(f"classifier training windows: {CLS_X.shape}")

classifier input: 86 channels x 97 frames
classifier training windows: (1430, 86, 97)


## 6. Table distance and side targets

Two things the original run omitted.

`window_at` passed zeros where Stage 5 had `table_dist`, which is the strongest
single feature for defensive play. And the detector's side head was never given
a loss, despite being used at inference to pick which player to classify.

In [7]:
# --- table distance per stream frame ---------------------------------------
for v in VIDEOS:
    d = S[v]
    raw = np.load(STREAM/f"{v}.npz", allow_pickle=True)
    tb = raw["table_box"]
    KPr = raw["keypoints"].astype(np.float32)
    hip = (KPr[:, :, L_HIP] + KPr[:, :, R_HIP]) / 2
    sho = (KPr[:, :, L_SHO] + KPr[:, :, R_SHO]) / 2
    torso = np.linalg.norm(sho - hip, axis=-1)
    td = np.zeros((len(hip), 2), np.float32)
    if tb[2] > tb[0]:
        for pi, edge in ((0, tb[0]), (1, tb[2])):
            t = torso[:, pi]
            med = np.median(t[t > 1]) if (t > 1).any() else 1.0
            td[:, pi] = np.abs(hip[:, pi, 0] - edge) / max(med, 1e-3)
    d["td"] = np.nan_to_num(td)

# --- side targets, masked outside contact neighbourhoods --------------------
for v in VIDEOS:
    d = S[v]
    ys = np.full(len(d["fidx"]), -1.0, np.float32)
    for i in d["pos"]:
        f = int(d["fidx"][i])
        ys[max(0, i-2):i+3] = 0.0 if GT[v][f][1] == "left" else 1.0
    d["ys"] = ys


def window_at(v, idx, side):
    """97-frame classifier input. Channel layout matches Stage 5 exactly:
    kp(34) vel(34) valid(17) table_dist(1) = 86."""
    d = S[v]
    sel = np.clip(np.arange(idx-PRE, idx-PRE+NF), 0, len(d["X"])-1)
    pi = 0 if side == "left" else 1
    kp = d["kp"][sel, pi]
    val = d["val"][sel, pi].astype(np.float32)
    vel = np.zeros_like(kp); vel[1:] = np.diff(kp, axis=0)
    td = d["td"][sel, pi][:, None]
    x = np.concatenate([kp.reshape(NF, -1), vel.reshape(NF, -1), val, td], 1)
    return np.nan_to_num(x).T.astype(np.float32)


# rebuild the classifier training set with the corrected windows
# GT holds (shot_class, side); technique comes from the stroke table
TECH_AT = {(r.video_id, int(r.frame_120)): r.technique
           for r in strokes.itertuples() if r.video_id in GT}

CLS_X, CLS_Y, CLS_T, CLS_V = [], [], [], []
for v in VIDEOS:
    for i in S[v]["pos"]:
        f = int(S[v]["fidx"][i])
        cls, side = GT[v][f]                      # 2-tuple, not 3
        tech = TECH_AT.get((v, f), "")
        CLS_X.append(window_at(v, i, side)); CLS_Y.append(C2I[cls])
        CLS_T.append(T2I.get(tech, -1)); CLS_V.append(v)

CLS_X = np.stack(CLS_X); CLS_Y = np.array(CLS_Y)
CLS_T = np.array(CLS_T); CLS_V = np.array(CLS_V)
CLS_F = np.array([V2F[v] for v in CLS_V])
C_CLS = CLS_X.shape[1]
print(f"classifier windows {CLS_X.shape}  ({C_CLS} channels, table_dist restored)")
print(f"technique labels resolved: {(CLS_T >= 0).sum()} of {len(CLS_T)}")

classifier windows (1430, 86, 97)  (86 channels, table_dist restored)
technique labels resolved: 1430 of 1430


## 7. Training loops

In [8]:
def train_det(f, rng):
    tr = [v for v in VIDEOS if S[v]["fold"] != f]
    cat = np.concatenate([S[v]["X"] for v in tr])
    mu, sd = cat.mean(0), cat.std(0) + 1e-6; del cat
    eff = np.mean([(S[v]["yc"] > 0.05).mean() for v in tr])
    pw = torch.tensor((1-eff)/eff, device=dev)
    net = DetNet(C_DET).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
    steps = DET_EPOCHS*40
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
    net.train()
    for _ in range(steps):
        xs, ys = [], []
        for _ in range(16):
            v = tr[rng.integers(len(tr))]; d = S[v]
            a, b = d["spans"][rng.integers(len(d["spans"]))]
            n = b - a
            if n <= 512:
                sl = slice(a, b); pad = 512 - n
            else:
                st = a + rng.integers(n-512+1); sl = slice(st, st+512); pad = 0
            x = (d["X"][sl]-mu)/sd; y = d["yc"][sl]
            if pad:
                x = np.pad(x, ((0,pad),(0,0)), mode="edge"); y = np.pad(y, (0,pad))
            xs.append(x.T); ys.append(y)
        x = torch.tensor(np.stack(xs), dtype=torch.float32, device=dev)
        y = torch.tensor(np.stack(ys), dtype=torch.float32, device=dev)
        lc, _ = net(x)
        loss = F.binary_cross_entropy_with_logits(lc, y, pos_weight=pw)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sch.step()
    return net, mu, sd


def train_cls(f):
    tr = CLS_F != f
    xt = torch.tensor(CLS_X[tr], device=dev)
    mu, sd = xt.mean((0,2), keepdim=True), xt.std((0,2), keepdim=True)+1e-6
    xt = (xt-mu)/sd
    yt = torch.tensor(CLS_Y[tr], device=dev)
    tt = torch.tensor(CLS_T[tr], device=dev)
    cnt = np.bincount(CLS_Y[tr], minlength=4).clip(1)
    cw = torch.tensor(len(CLS_Y[tr])/(4*cnt), dtype=torch.float32, device=dev)
    p = (1.0/cnt)[CLS_Y[tr]]; p = p/p.sum()
    net = ClsNet(C_CLS).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
    steps = CLS_EPOCHS*max(1, math.ceil(tr.sum()/BATCH))
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
    net.train()
    for _ in range(steps):
        b = np.random.choice(tr.sum(), BATCH, p=p)
        xb = xt[b]
        sh = torch.randint(-6, 7, (len(b),), device=dev)
        ix = (torch.arange(NF, device=dev)[None]+sh[:,None]).clamp(0, NF-1)
        xb = torch.gather(xb, 2, ix[:,None].expand(-1, xb.shape[1], -1))
        xb = xb + torch.randn_like(xb)*0.01
        s_, t_ = net(xb)
        loss = focal(s_, yt[b], cw) + 0.2*focal(t_, tt[b])
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sch.step()
    return net, mu, sd


def decode(prob, thr, gap=30):
    idx = np.where(prob >= thr)[0]
    if not len(idx): return np.array([], int)
    pk = [i for i in idx if prob[i] == prob[max(0,i-15):i+16].max()]
    pk = sorted(pk, key=lambda i: -prob[i]); kept = []
    for p in pk:
        if all(abs(p-k) >= gap for k in kept): kept.append(p)
    return np.array(sorted(kept), int)


seed_all(SEED); rng = np.random.default_rng(SEED)
PRED = {}
for f in FOLDS:
    va = [v for v in VIDEOS if S[v]["fold"] == f]
    if not va: continue
    dnet, dmu, dsd = train_det(f, rng)
    cnet, cmu, csd = train_cls(f)
    dnet.eval(); cnet.eval()
    for v in va:
        d = S[v]
        prob = np.zeros(len(d["X"]), np.float32)
        sidep = np.zeros(len(d["X"]), np.float32)
        with torch.no_grad():
            for a, b in d["spans"]:
                x = torch.tensor(((d["X"][a:b]-dmu)/dsd).T[None],
                                 dtype=torch.float32, device=dev)
                lc, ls = dnet(x)
                prob[a:b] = torch.sigmoid(lc)[0].cpu().numpy()
                sidep[a:b] = torch.sigmoid(ls)[0].cpu().numpy()
        pk = decode(prob, 0.60)
        if len(pk):
            sides = ["right" if sidep[i] >= 0.5 else "left" for i in pk]
            xb = torch.tensor(np.stack([window_at(v, i, s)
                                        for i, s in zip(pk, sides)]), device=dev)
            with torch.no_grad():
                lo, _ = cnet((xb-cmu)/csd)
                pr = torch.softmax(lo, 1).cpu().numpy()
        else:
            sides, pr = [], np.zeros((0, 4))
        PRED[v] = dict(peaks=pk, sides=sides, proba=pr, prob=prob)
    print(f"  fold {f}: {va} done")
print("\ncascade inference complete")

  fold A: ['game_1'] done
  fold B: ['game_2'] done
  fold C: ['game_3'] done
  fold D: ['game_4'] done
  fold E: ['game_5'] done
  fold F: ['test_1', 'test_4'] done
  fold G: ['test_2', 'test_3', 'test_6', 'test_7'] done

cascade inference complete


## 8. Detector training, with the side loss

The one-line omission that cost 0.31 macro-F1: the original loop computed only
the contact loss, so the side head stayed at its initialisation while inference
relied on it.

In [9]:
def train_det2(f, rng):
    tr = [v for v in VIDEOS if S[v]["fold"] != f]
    cat = np.concatenate([S[v]["X"] for v in tr])
    mu, sd = cat.mean(0), cat.std(0) + 1e-6; del cat
    eff = np.mean([(S[v]["yc"] > 0.05).mean() for v in tr])
    pw = torch.tensor((1-eff)/eff, device=dev)

    net = DetNet(C_DET).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
    steps = DET_EPOCHS*40
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
    net.train()
    for _ in range(steps):
        xs, ys_, ss = [], [], []
        for _ in range(16):
            v = tr[rng.integers(len(tr))]; d = S[v]
            a, b = d["spans"][rng.integers(len(d["spans"]))]
            n = b - a
            if n <= 512:
                sl = slice(a, b); pad = 512 - n
            else:
                st = a + rng.integers(n-512+1); sl = slice(st, st+512); pad = 0
            x = (d["X"][sl]-mu)/sd; y = d["yc"][sl]; s_ = d["ys"][sl]
            if pad:
                x = np.pad(x, ((0,pad),(0,0)), mode="edge")
                y = np.pad(y, (0,pad)); s_ = np.pad(s_, (0,pad), constant_values=-1)
            xs.append(x.T); ys_.append(y); ss.append(s_)
        x = torch.tensor(np.stack(xs), dtype=torch.float32, device=dev)
        y = torch.tensor(np.stack(ys_), dtype=torch.float32, device=dev)
        s_ = torch.tensor(np.stack(ss), dtype=torch.float32, device=dev)

        lc, ls = net(x)
        loss = F.binary_cross_entropy_with_logits(lc, y, pos_weight=pw)
        m = (s_ >= 0).float()
        if m.sum() > 0:                       # <-- the missing term
            loss = loss + 0.3*(F.binary_cross_entropy_with_logits(
                ls, s_.clamp(min=0), reduction="none")*m).sum()/m.sum()

        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sch.step()
    return net, mu, sd

print("train_det2 ready - side loss included")

train_det2 ready — side loss included


## 9. Matching

In [10]:
def match_pairs(pred, true, tol):
    used, pairs = set(), []
    for k, p in enumerate(pred):
        best, bd = None, tol+1
        for j, t in enumerate(true):
            if j in used: continue
            dist = abs(p-t)
            if dist <= tol and dist < bd: best, bd = j, dist
        if best is not None:
            used.add(best); pairs.append((k, best))
    return pairs

from sklearn.metrics import f1_score

rows = []
for tol in TOLS:
    dTP = dFP = dFN = 0
    yt_all, yp_all = [], []
    for v in VIDEOS:
        d, P = S[v], PRED[v]
        true = d["pos"]
        pairs = match_pairs(P["peaks"], true, tol)
        dTP += len(pairs); dFP += len(P["peaks"])-len(pairs); dFN += len(true)-len(pairs)
        for k, j in pairs:
            f = int(d["fidx"][true[j]])
            yt_all.append(C2I[GT[v][f][0]])
            yp_all.append(int(P["proba"][k].argmax()))
    dp = dTP/max(dTP+dFP,1); dr = dTP/max(dTP+dFN,1)
    df1 = 2*dp*dr/max(dp+dr,1e-9)
    cls_acc = float(np.mean(np.array(yt_all) == np.array(yp_all))) if yt_all else 0.0
    cls_f1 = f1_score(yt_all, yp_all, average="macro", zero_division=0) if yt_all else 0.0
    # end-to-end: correct only if detected AND classified right
    e2e_tp = sum(1 for a, b in zip(yt_all, yp_all) if a == b)
    ep = e2e_tp/max(dTP+dFP,1); er = e2e_tp/max(dTP+dFN,1)
    e2e_f1 = 2*ep*er/max(ep+er,1e-9)
    rows.append(dict(tol=tol, ms=round(tol/120*1000), det_f1=df1,
                     cls_acc=cls_acc, cls_macro_f1=cls_f1, e2e_f1=e2e_f1,
                     n=len(yt_all)))

sweep = pd.DataFrame(rows)
sweep.to_csv(OUT/"metrics/stage7_tolerance_sweep.csv", index=False)
print("=" * 78)
print("TOLERANCE SWEEP")
print("=" * 78)
print(sweep.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

base = float(sweep[sweep.tol == 20].cls_acc.iloc[0])
print("\n" + "=" * 78)
print("DOES THE CLASSIFIER CARE ABOUT DETECTION JITTER?")
print("=" * 78)
print(f"  classifier accuracy on windows centred at the loosest match "
      f"(+/-20): {base:.3f}")
for _, r in sweep.iterrows():
    print(f"    +/-{int(r.tol):<3} ({int(r.ms):>3} ms): acc {r.cls_acc:.3f}   "
          f"{r.cls_acc-base:+.3f}")

a3 = float(sweep[sweep.tol == 3].cls_acc.iloc[0])
a8 = float(sweep[sweep.tol == 8].cls_acc.iloc[0])
drop = a3 - a8
print(f"\n  accuracy at +/-3 vs +/-8: {a3:.3f} vs {a8:.3f}  ({drop:+.3f})")
if abs(drop) < 0.03:
    print("""
  FLAT. Tightening the tolerance from 8 frames to 3 does not make the
  classifier more accurate, so the extra timing precision buys nothing
  downstream. +/-8 (67 ms) is the operationally correct bar, and detection
  at that tolerance is fine.""")
else:
    print(f"""
  NOT FLAT ({drop:+.3f}). Timing precision does matter to the classifier,
  so +/-5 was a reasonable bar after all and detection IS the bottleneck.
  Improving contact localisation is the highest-value next step.""")
print("=" * 78)

TOLERANCE SWEEP
 tol  ms  det_f1  cls_acc  cls_macro_f1  e2e_f1    n
   1   8   0.379    0.513         0.477   0.195  526
   2  17   0.562    0.500         0.470   0.281  780
   3  25   0.676    0.490         0.471   0.331  938
   5  42   0.797    0.495         0.488   0.395 1106
   8  67   0.855    0.491         0.488   0.420 1187
  12 100   0.870    0.492         0.490   0.428 1208
  20 167   0.881    0.493         0.492   0.434 1223

DOES THE CLASSIFIER CARE ABOUT DETECTION JITTER?
  classifier accuracy on windows centred at the loosest match (+/-20): 0.493
    +/-1   (  8 ms): acc 0.513   +0.020
    +/-2   ( 17 ms): acc 0.500   +0.007
    +/-3   ( 25 ms): acc 0.490   -0.003
    +/-5   ( 42 ms): acc 0.495   +0.002
    +/-8   ( 67 ms): acc 0.491   -0.002
    +/-12  (100 ms): acc 0.492   -0.001
    +/-20  (167 ms): acc 0.493   +0.000

  accuracy at +/-3 vs +/-8: 0.490 vs 0.491  (-0.001)

  FLAT. Tightening the tolerance from 8 frames to 3 does not make the
  classifier more accurate, 

## 10. Run the cascade

Classifier windows are cut around **predicted** contacts, so detection error
propagates. That is the point of the measurement.

In [11]:
seed_all(SEED); rng = np.random.default_rng(SEED)
PRED, side_stats = {}, []

for f in FOLDS:
    va = [v for v in VIDEOS if S[v]["fold"] == f]
    if not va: continue
    dnet, dmu, dsd = train_det2(f, rng)
    cnet, cmu, csd = train_cls(f)
    dnet.eval(); cnet.eval()

    for v in va:
        d = S[v]
        prob = np.zeros(len(d["X"]), np.float32)
        sidep = np.zeros(len(d["X"]), np.float32)
        with torch.no_grad():
            for a, b in d["spans"]:
                x = torch.tensor(((d["X"][a:b]-dmu)/dsd).T[None],
                                 dtype=torch.float32, device=dev)
                lc, ls = dnet(x)
                prob[a:b] = torch.sigmoid(lc)[0].cpu().numpy()
                sidep[a:b] = torch.sigmoid(ls)[0].cpu().numpy()
        pk = decode(prob, 0.60)
        sides = ["right" if sidep[i] >= 0.5 else "left" for i in pk]

        ok = tot = 0
        for k, i in enumerate(pk):
            if not len(d["pos"]): continue
            j = np.abs(d["pos"]-i).argmin()
            if abs(d["pos"][j]-i) > 8: continue
            tot += 1
            ok += int(sides[k] == GT[v][int(d["fidx"][d["pos"][j]])][1])
        side_stats.append(dict(video_id=v, n=tot, side_acc=ok/max(tot, 1)))

        if len(pk):
            xb = torch.tensor(np.stack([window_at(v, i, s)
                                        for i, s in zip(pk, sides)]), device=dev)
            with torch.no_grad():
                lo, _ = cnet((xb-cmu)/csd)
                pr = torch.softmax(lo, 1).cpu().numpy()
        else:
            pr = np.zeros((0, 4))
        PRED[v] = dict(peaks=pk, sides=sides, proba=pr, prob=prob)
    print(f"  fold {f} done")

ss = pd.DataFrame(side_stats)
print(f"\nside-head accuracy: {(ss.side_acc*ss.n).sum()/ss.n.sum():.3f}")
print("  (it was ~0.50 before the fix, because the head was never trained)")

  fold A done
  fold B done
  fold C done
  fold D done
  fold E done
  fold F done
  fold G done

side-head accuracy: 0.994
  (it was ~0.50 before the fix, because the head was never trained)


## 11. Results, and save the confusion matrix

In [12]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_recall_fscore_support)

TOL = 8
yt_all, yp_all = [], []
per_video, tp_tot = [], 0

for v in VIDEOS:
    d, P = S[v], PRED[v]
    pairs = match_pairs(P["peaks"], d["pos"], TOL)
    yt = [C2I[GT[v][int(d["fidx"][d["pos"][j]])][0]] for _, j in pairs]
    yp_ = [int(P["proba"][k].argmax()) for k, _ in pairs]
    yt_all += yt; yp_all += yp_
    tp = sum(1 for a, b in zip(yt, yp_) if a == b); tp_tot += tp
    fp = len(P["peaks"]) - tp; fn = len(d["pos"]) - tp
    p_, r_ = tp/max(tp+fp, 1), tp/max(tp+fn, 1)
    per_video.append(dict(video_id=v, fold=d["fold"], n=len(d["pos"]),
                          detected=len(pairs), correct=tp,
                          precision=p_, recall=r_,
                          f1=2*p_*r_/max(p_+r_, 1e-9)))

yt_all, yp_all = np.array(yt_all), np.array(yp_all)

print("=" * 74)
print(f"END-TO-END CASCADE @ +/-{TOL} frames ({TOL/120*1000:.0f} ms)")
print("=" * 74)
print(classification_report(yt_all, yp_all, target_names=CLASSES,
                            digits=3, zero_division=0))

cm = confusion_matrix(yt_all, yp_all, labels=range(4))
cmdf = pd.DataFrame(cm, index=[f"true_{c}" for c in CLASSES],
                    columns=[f"pred_{c}" for c in CLASSES])
cmdf["recall"] = (np.diag(cm)/cm.sum(1)).round(3)
print(cmdf.to_string())

# --- save what the original run failed to ----------------------------------
cmdf.to_csv(OUT/"metrics/stage7_v2_confusion.csv")

pr, rc, f1, sup = precision_recall_fscore_support(
    yt_all, yp_all, labels=range(4), zero_division=0)
pc = pd.DataFrame(dict(cls=CLASSES, precision=pr, recall=rc, f1=f1,
                       support=sup))
pc.loc[len(pc)] = ["macro", pr.mean(), rc.mean(),
                   f1_score(yt_all, yp_all, average="macro", zero_division=0),
                   sup.sum()]
pc.to_csv(OUT/"metrics/stage7_v2_per_class.csv", index=False)

pv = pd.DataFrame(per_video)
pv.to_csv(OUT/"metrics/stage7_v2_per_video.csv", index=False)
print("\n" + pv.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

ALL_P = sum(len(PRED[v]["peaks"]) for v in VIDEOS)
ALL_T = sum(len(S[v]["pos"]) for v in VIDEOS)
p_, r_ = tp_tot/max(ALL_P, 1), tp_tot/max(ALL_T, 1)
e2e = 2*p_*r_/max(p_+r_, 1e-9)
macro_cls = f1_score(yt_all, yp_all, average="macro", zero_division=0)

print("\n" + "=" * 74)
print(f"  classification macro-F1 (on detected windows)  {macro_cls:.3f}")
print(f"  end-to-end F1                                  {e2e:.3f}")
print(f"\n  pre-fix run reported 0.481 - that was the untrained side head,")
print(f"  not the cost of cascading.")
print("=" * 74)

print(f"""
SAVED
  stage7_v2_confusion.csv    per-class confusion, corrected
  stage7_v2_per_class.csv    precision / recall / F1 / support
  stage7_v2_per_video.csv    per-video breakdown

NEXT
  In 17_results.ipynb, Table 4 should read stage7_v2_confusion.csv rather than
  stage7_confusion.csv. Change:

      casc = load("stage7_confusion.csv")

  to:

      casc = first_of("stage7_v2_confusion.csv", "stage7_confusion.csv")

  Then delete stage7_confusion.csv so the stale file cannot be picked up by
  anything else.""")

END-TO-END CASCADE @ +/-8 frames (67 ms)
              precision    recall  f1-score   support

       serve      0.988     0.972     0.980       249
      attack      0.853     0.789     0.820       568
     control      0.778     0.766     0.772       256
     defence      0.400     0.532     0.457       154

    accuracy                          0.789      1227
   macro avg      0.755     0.765     0.757      1227
weighted avg      0.808     0.789     0.797      1227

              pred_serve  pred_attack  pred_control  pred_defence  recall
true_serve           242            3             3             1   0.972
true_attack            1          448            35            84   0.789
true_control           1           21           196            38   0.766
true_defence           1           53            18            82   0.532

video_id fold   n  detected  correct  precision  recall    f1
  game_1    A 161       139       99      0.647   0.615 0.631
  game_2    B 399       327  

## 12. Tolerance sweep

In [13]:
rows = []
for tol in (1, 2, 3, 5, 8, 12, 20):
    dTP = dFP = dFN = 0; ya, yb = [], []
    for v in VIDEOS:
        d, P = S[v], PRED[v]
        pairs = match_pairs(P["peaks"], d["pos"], tol)
        dTP += len(pairs); dFP += len(P["peaks"])-len(pairs)
        dFN += len(d["pos"])-len(pairs)
        for k, j in pairs:
            ya.append(C2I[GT[v][int(d["fidx"][d["pos"][j]])][0]])
            yb.append(int(P["proba"][k].argmax()))
    dp = dTP/max(dTP+dFP, 1); dr = dTP/max(dTP+dFN, 1)
    acc = float(np.mean(np.array(ya) == np.array(yb))) if ya else 0.0
    cf1 = f1_score(ya, yb, average="macro", zero_division=0) if ya else 0.0
    tp = sum(1 for a, b in zip(ya, yb) if a == b)
    ep, er = tp/max(dTP+dFP, 1), tp/max(dTP+dFN, 1)
    rows.append(dict(tol=tol, ms=round(tol/120*1000),
                     det_f1=2*dp*dr/max(dp+dr, 1e-9),
                     cls_acc=acc, cls_macro_f1=cf1,
                     e2e_f1=2*ep*er/max(ep+er, 1e-9)))
sweep = pd.DataFrame(rows)
sweep.to_csv(OUT/"metrics/stage7_v2_sweep.csv", index=False)
print(sweep.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

a3 = float(sweep.loc[sweep.tol == 3, "cls_acc"].iloc[0])
a8 = float(sweep.loc[sweep.tol == 8, "cls_acc"].iloc[0])
print(f"""
  Classifier accuracy moves {a3:.3f} -> {a8:.3f} ({a3-a8:+.3f}) across a
  factor-of-two change in detection tolerance, so +/-8 frames is the
  operationally correct bar rather than the +/-5 chosen a priori.

  Measured rather than argued: windows were cut around PREDICTED contacts, so
  the jitter was real and a sharp degradation would have shown.""")

 tol  ms  det_f1  cls_acc  cls_macro_f1  e2e_f1
   1   8   0.409    0.820         0.763   0.335
   2  17   0.601    0.809         0.753   0.486
   3  25   0.715    0.805         0.756   0.575
   5  42   0.827    0.794         0.756   0.657
   8  67   0.879    0.789         0.757   0.693
  12 100   0.890    0.788         0.759   0.702
  20 167   0.900    0.789         0.760   0.710

  Classifier accuracy moves 0.805 -> 0.789 (+0.016) across a
  factor-of-two change in detection tolerance, so +/-8 frames is the
  operationally correct bar rather than the +/-5 chosen a priori.

  Measured rather than argued: windows were cut around PREDICTED contacts, so
  the jitter was real and a sharp degradation would have shown.


---
## After this

1. Point Table 4 in `17_results.ipynb` at `stage7_v2_confusion.csv`
2. Delete `stage7_confusion.csv`, so nothing can pick up the pre-fix run again

The corrected cascade number is the one that belongs in the paper. The 0.481
figure was a bug in which roughly half the classifier windows were cut around
the wrong player, not a measurement of anything.
